# Classification of messages as spam or not spam using Naive Bayes algorithm

**Import Dataset - upload the SMS text file to the content folder on the left panel before running**

In [1]:
import pandas as pd
import numpy as np

# Import Dataset - upload the SMS text file to the content folder on the left panel before running
df = pd.read_table('SMS.txt', sep='\t', header=None, names=['label', 'sms_message'])
df

,label,sms_message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [2]:
# map the 'ham' value to 0 and the 'spam' value to 1.
df['label_binary'] = df.label.map({'ham':0,'spam':1})
df.head()

,label,sms_message,label_binary
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [3]:
# Get stats
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
#  data cleaning
df['sms_message'] = df['sms_message'].str.replace(r'[\W_]+', ' ', regex=True).str.strip() # Removes punctuation and leading/trailing spaces
df['sms_message'] = df['sms_message'].str.lower() ### making all the words lowercase
df.head(10)

,label,sms_message,label_binary
0,ham,go until jurong point crazy available only in ...,0
1,ham,ok lar joking wif u oni,0
2,spam,free entry in 2 a wkly comp to win fa cup fina...,1
3,ham,u dun say so early hor u c already then say,0
4,ham,nah i don t think he goes to usf he lives arou...,0
5,spam,freemsg hey there darling it s been 3 week s n...,1
6,ham,even my brother is not like to speak with me t...,0
7,ham,as per your request melle melle oru minnaminun...,0
8,spam,winner as a valued network customer you have b...,1
9,spam,had your mobile 11 months or more u r entitled...,1


In [5]:
# Randomly shuffle the records in the dataset to avoid bias
df = df.sample(frac=1, random_state=1)
df.head(10)

,label,sms_message,label_binary
1078,ham,yep by the pretty sculpture,0
4028,ham,yes princess are you going to make me moan,0
958,ham,welp apparently he retired,0
4642,ham,havent,0
4674,ham,i forgot 2 ask ü all smth there s a card on da...,0
5461,ham,ok i thk i got it then u wan me 2 come now or wat,0
4210,ham,i want kfc its tuesday only buy 2 meals only 2...,0
4216,ham,no dear i was sleeping p,0
1603,ham,ok pa nothing problem,0
1504,ham,ill be there on lt gt ok,0


In [6]:
# Split into training and test sets
training_test_index = round(len(df) * 0.8)

training = df[:training_test_index].reset_index(drop=True)
test = df[training_test_index:].reset_index(drop=True)

print('-- Training set stats --')
print(training.shape)
print(training['label_binary'].value_counts())
print('-- Test set stats --')
print(test.shape)
print(test['label_binary'].value_counts())

-- Training set stats --
(4458, 3)
label_binary
0    3858
1     600
Name: count, dtype: int64
-- Test set stats --
(1114, 3)
label_binary
0    967
1    147
Name: count, dtype: int64


In [7]:
### creating vocabulary from training data
training['sms_message'] = training['sms_message'].str.split()
vocabulary = []
for sms in training['sms_message']:
   for word in sms:
      vocabulary.append(word)
vocabulary = list(set(vocabulary))  ### only count the number of unique words
print(len(vocabulary))
vocabulary[0:9]

7780


['raise',
 'woo',
 'tacos',
 'deeraj',
 'virgins',
 '08715203677',
 'interview',
 'postcode',
 '447797706009']

In [8]:
word_counts_per_sms = {unique_word: [0] * len(training['sms_message']) for unique_word in vocabulary}

for index, sms in enumerate(training['sms_message']):
   for word in sms:
      word_counts_per_sms[word][index] += 1
word_counts = pd.DataFrame(word_counts_per_sms)
word_counts

,raise,woo,tacos,deeraj,virgins,08715203677,interview,postcode,447797706009,fullonsms,...,alert,blimey,09064017295,21,alone,snickering,minecraft,timin,bfore,single
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4453,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4454,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4455,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4456,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
training_new = pd.concat([training, word_counts], axis=1)
training_new.head()

,label,sms_message,label_binary,raise,woo,tacos,deeraj,virgins,08715203677,interview,...,alert,blimey,09064017295,21,alone,snickering,minecraft,timin,bfore,single
0,ham,"[yep, by, the, pretty, sculpture]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ham,"[yes, princess, are, you, going, to, make, me,...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ham,"[welp, apparently, he, retired]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ham,[havent],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ham,"[i, forgot, 2, ask, ü, all, smth, there, s, a,...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
# Run a baseline model evaluation
# Set all 'predicted to 0 or 1 randomly to get a baseline (coin-flip)
test['predicted'] = np.random.randint(0, 2, size=len(test))
test['predicted'].value_counts()

predicted
0    579
1    535
Name: count, dtype: int64

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print('Accuracy score: {}'.format(accuracy_score(test['label_binary'], test['predicted'])))
print('Precision score: {}'.format(precision_score(test['label_binary'], test['predicted'])))
print('Recall score: {}'.format(recall_score(test['label_binary'], test['predicted'])))
print('F1 score: {}'.format(f1_score(test['label_binary'], test['predicted'])))

Accuracy score: 0.5278276481149012
Precision score: 0.14579439252336449
Recall score: 0.5306122448979592
F1 score: 0.2287390029325513


# **My implementation starts here**.


Make sure your prediction result is saved into the column `test['predicted']` for the evaluation to run automatically.  
**50 points** for successful execution of your code and producing the confusion matrix correctly

In [12]:
# Laplace smoothing
alpha = 1

In [13]:
import math
import re

# priors
spam_df = training[training['label_binary'] == 1]
ham_df = training[training['label_binary'] == 0]
p_spam = len(spam_df) / len(training)
p_ham = len(ham_df) / len(training)
print('P(spam) =', round(p_spam, 4))
print('P(ham) =', round(p_ham, 4))

# total words in spam / ham messages
n_spam = 0
for sms in spam_df['sms_message']:
    n_spam += len(sms)
n_ham = 0
for sms in ham_df['sms_message']:
    n_ham += len(sms)
n_vocab = len(vocabulary)

# per-word counts in each class (sum across the right rows of word_counts)
n_w_spam = word_counts.loc[spam_df.index].sum(axis=0)
n_w_ham = word_counts.loc[ham_df.index].sum(axis=0)

# laplace smoothed probabilities
denom_spam = n_spam + alpha * n_vocab
denom_ham = n_ham + alpha * n_vocab

p_w_spam = {}
p_w_ham = {}
for w in vocabulary:
    p_w_spam[w] = (n_w_spam[w] + alpha) / denom_spam
    p_w_ham[w] = (n_w_ham[w] + alpha) / denom_ham

# use logs so multiplying lots of small numbers doesn't underflow
log_p_spam = math.log(p_spam)
log_p_ham = math.log(p_ham)
log_pw_spam = {}
log_pw_ham = {}
for w in vocabulary:
    log_pw_spam[w] = math.log(p_w_spam[w])
    log_pw_ham[w] = math.log(p_w_ham[w])

def classify(message):
    message = re.sub(r'[\W_]+', ' ', message).lower().strip()
    words = message.split()
    s = log_p_spam
    h = log_p_ham
    for w in words:
        if w in log_pw_spam:
            s = s + log_pw_spam[w]
            h = h + log_pw_ham[w]
    if s > h:
        return 1
    else:
        return 0

test['predicted'] = test['sms_message'].apply(classify)

# confusion matrix
TP = 0
TN = 0
FP = 0
FN = 0
for i in range(len(test)):
    actual = test['label_binary'].iloc[i]
    pred = test['predicted'].iloc[i]
    if actual == 1 and pred == 1:
        TP += 1
    elif actual == 0 and pred == 0:
        TN += 1
    elif actual == 0 and pred == 1:
        FP += 1
    elif actual == 1 and pred == 0:
        FN += 1

print()
print('Confusion Matrix')
print('             pred ham   pred spam')
print('actual ham   TN =', TN, '   FP =', FP)
print('actual spam  FN =', FN, '   TP =', TP)
print('total:', TP + TN + FP + FN)

print()
print('label_binary counts:')
print(test['label_binary'].value_counts())
print('predicted counts:')
print(test['predicted'].value_counts())

P(spam) = 0.1346
P(ham) = 0.8654

Confusion Matrix
             pred ham   pred spam
actual ham   TN = 962    FP = 5
actual spam  FN = 8    TP = 139
total: 1114

label_binary counts:
label_binary
0    967
1    147
Name: count, dtype: int64
predicted counts:
predicted
0    970
1    144
Name: count, dtype: int64


**Evaluate your implementation** for accuracy, precision, recall and F1_score.  The performance points of your implementation will be calculated automatically.  However, it is only awarded if the predictions are made by a Naive Bayes implementation.

**30 points** for how well your implementation predicts spam.  A correct implementation should achieve an F1 score above 0.90.  
## **DO NOT modify this cell below.**

In [14]:
# Model Evaluation
print('Accuracy score: {}'.format(accuracy_score(test['label_binary'], test['predicted'])))
print('Precision score: {}'.format(precision_score(test['label_binary'], test['predicted'])))
print('Recall score: {}'.format(recall_score(test['label_binary'], test['predicted'])))
my_f1_score = f1_score(test['label_binary'], test['predicted'])
print('F1 score: {}'.format(my_f1_score))
performance_point = round(np.clip((my_f1_score - 0.20) / (0.9-0.20) * 30, 0, 30))
print('Your performance point: {}'.format(performance_point))

Accuracy score: 0.9883303411131059
Precision score: 0.9652777777777778
Recall score: 0.9455782312925171
F1 score: 0.9553264604810997
Your performance point: 30


**Analyze your implementation of the Naive Bayes algorithm:** select an entry from each quadrant of the confusion matrix and show the details of the prediction, i.e., the probability of being a spam or a ham, and all the contributing probabilities.  Discuss why mis-classification ocurrs for the FP and FN examples.

**20 points** for a correct and clear presentation.

In [15]:
# pick one example from each quadrant and print the probabilities for each word

def explain(message, true_label):
    cleaned = re.sub(r'[\W_]+', ' ', message).lower().strip()
    words = cleaned.split()
    s = log_p_spam
    h = log_p_ham

    if true_label == 1:
        print('actual: spam')
    else:
        print('actual: ham')
    print('message:', message)
    print('prior log P(spam) =', round(log_p_spam, 4), ' prior log P(ham) =', round(log_p_ham, 4))
    print('word   P(w|spam)   P(w|ham)')
    for w in words:
        if w in p_w_spam:
            ps = p_w_spam[w]
            ph = p_w_ham[w]
            s = s + math.log(ps)
            h = h + math.log(ph)
            print(w, '  ', ps, '  ', ph)
        else:
            print(w, '  (not in vocab, skipped)')

    print('log P(spam | msg) =', round(s, 3))
    print('log P(ham | msg)  =', round(h, 3))
    if s > h:
        pred = 1
        print('prediction: spam')
    else:
        pred = 0
        print('prediction: ham')
    if pred == true_label:
        print('--> correct')
    else:
        print('--> wrong')
    print()

tp_idx = test[(test['label_binary'] == 1) & (test['predicted'] == 1)].index
tn_idx = test[(test['label_binary'] == 0) & (test['predicted'] == 0)].index
fp_idx = test[(test['label_binary'] == 0) & (test['predicted'] == 1)].index
fn_idx = test[(test['label_binary'] == 1) & (test['predicted'] == 0)].index

print('--- True Positive (spam, predicted spam) ---')
i = tp_idx[0]
explain(test['sms_message'].loc[i], test['label_binary'].loc[i])

print('--- True Negative (ham, predicted ham) ---')
i = tn_idx[0]
explain(test['sms_message'].loc[i], test['label_binary'].loc[i])

print('--- False Positive (ham, predicted spam) ---')
if len(fp_idx) > 0:
    i = fp_idx[0]
    explain(test['sms_message'].loc[i], test['label_binary'].loc[i])
else:
    print('no false positives')

print('--- False Negative (spam, predicted ham) ---')
if len(fn_idx) > 0:
    i = fn_idx[0]
    explain(test['sms_message'].loc[i], test['label_binary'].loc[i])
else:
    print('no false negatives')

--- True Positive (spam, predicted spam) ---
actual: spam
message: had your mobile 10 mths update to latest orange camera video phones for free save s with free texts weekend calls text yes for a callback orno to opt out
prior log P(spam) = -2.0055  prior log P(ham) = -0.1446
word   P(w|spam)   P(w|ham)
had    0.00047882296609062813    0.0011536154307600019
your    0.009228224437383015    0.005229723286112009
mobile    0.00426587733426196    0.00021534154707520035
10    0.0009576459321812563    0.0001845784689216003
mths    0.0001305880816610804    3.076307815360005e-05
update    0.00047882296609062813    7.690769538400013e-05
to    0.023810560222870324    0.019596080783843232
latest    0.001218822095503417    4.6144617230400075e-05
orange    0.0010011752927349498    7.690769538400013e-05
camera    0.0010447046532886433    4.6144617230400075e-05
video    0.0012623514560571106    4.6144617230400075e-05
phones    0.0003482348844295477    3.076307815360005e-05
for    0.006703521525268794 

**Discussion**

The interesting cases are the ones the classifier gets wrong. The FP message "unlimited texts limited minutes" only has three words that are actually in the vocab, and two of them ("unlimited" and "texts") happen to be way more common in spam than in ham in the training set. Since the message is so short there's nothing else to balance it out, so the spam score edges out the ham score even though a person reading it would clearly say this is a normal text. This is the big weakness of short messages — one or two spam-leaning words and the bag-of-words view tips the wrong way.

The FN is the opposite problem. It's a long spam message but a lot of what makes it obviously spam — the phone number, "knickers", "nikiyu4" — isn't in the vocab and gets thrown out. What's left is mostly everyday words like "me", "my", "it", "u", which in the training data show up more in ham than in spam. So the model ends up scoring it as ham. Naive Bayes basically loses anything it hasn't seen before, and it also can't use word order, so a spam message that's written to sound conversational can slip past.

For the correctly classified ones it's more straightforward. The TP is long and packed with spammy words ("free" twice, "mobile", "text", "calls", "opt", "latest"), each of which pushes the spam score up by a noticeable amount, so by the end the spam side wins by a huge margin (log P(spam|msg) ≈ -189 vs log P(ham|msg) ≈ -224). The TN is the boring success case — a short normal message where every word is more common in ham than in spam, so the ham prior plus the ham-heavy words makes it an easy call.

One thing worth noting: the prior is very skewed (P(ham) ≈ 0.865, P(spam) ≈ 0.135), so every message starts off biased toward ham. The spam predictions only happen when enough per-word evidence piles up to overcome that gap, which is also why the FP is so close — the spam score only beats ham by about 2 in log space on that one.

**Bonus for 20 points:**  Use function MultinomialNB (from sklearn.naive_bayes import MultinomialNB) to perform the same classification and evaludate its results.

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# CountVectorizer expects strings, but training['sms_message'] was split earlier,
# so join the token lists back into strings
train_texts = []
for toks in training['sms_message']:
    train_texts.append(' '.join(toks))
test_texts = test['sms_message']

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)
y_train = training['label_binary']
y_test = test['label_binary']

sk_model = MultinomialNB(alpha=alpha)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_test)

# confusion matrix for sklearn
sk_TP = 0
sk_TN = 0
sk_FP = 0
sk_FN = 0
for i in range(len(y_test)):
    actual = y_test.iloc[i]
    pred = sk_pred[i]
    if actual == 1 and pred == 1:
        sk_TP += 1
    elif actual == 0 and pred == 0:
        sk_TN += 1
    elif actual == 0 and pred == 1:
        sk_FP += 1
    elif actual == 1 and pred == 0:
        sk_FN += 1

print('MultinomialNB confusion matrix')
print('             pred ham   pred spam')
print('actual ham   TN =', sk_TN, '   FP =', sk_FP)
print('actual spam  FN =', sk_FN, '   TP =', sk_TP)

print()
print('MultinomialNB:')
print('  accuracy :', round(accuracy_score(y_test, sk_pred), 4))
print('  precision:', round(precision_score(y_test, sk_pred), 4))
print('  recall   :', round(recall_score(y_test, sk_pred), 4))
print('  f1       :', round(f1_score(y_test, sk_pred), 4))

print()
print('My implementation:')
print('  accuracy :', round(accuracy_score(test['label_binary'], test['predicted']), 4))
print('  precision:', round(precision_score(test['label_binary'], test['predicted']), 4))
print('  recall   :', round(recall_score(test['label_binary'], test['predicted']), 4))
print('  f1       :', round(f1_score(test['label_binary'], test['predicted']), 4))

# scores are close, mine comes out a little higher

MultinomialNB confusion matrix
             pred ham   pred spam
actual ham   TN = 959    FP = 8
actual spam  FN = 10    TP = 137

MultinomialNB:
  accuracy : 0.9838
  precision: 0.9448
  recall   : 0.932
  f1       : 0.9384

My implementation:
  accuracy : 0.9883
  precision: 0.9653
  recall   : 0.9456
  f1       : 0.9553
